In [ ]:
import rdflib
from rdflib.namespace import OWL
from collections import defaultdict
from pathlib import Path


## MERGE EQUIVALENT IDENTIFIERS INTO CELEX ID
def merge_identifiers(input_path, output_path):

    g = rdflib.Graph()
    g.parse(input_path, format='xml')

    # Disjoint Set Union (DSU)
    parent = {}
    def find(node):
        parent.setdefault(node, node)
        root = node
        while parent[root] != root:
            root = parent[root]
        while parent[node] != root:
            parent[node], node = root, parent[node]
        return root

    def union(a, b):
        root_a, root_b = find(a), find(b)
        if root_a != root_b:
            parent[root_b] = root_a

    for s, p, o in g.triples((None, OWL.sameAs, None)):
        union(s, o)
        
    # Collect the nodes into their clusters
    clusters = defaultdict(set)
    for node in list(parent):
        clusters[find(node)].add(node)

    replace_map = {}
    for cluster_nodes in clusters.values():
        celex_uris = sorted(n for n in cluster_nodes if "/celex/" in str(n))
        canonical = celex_uris[0] if celex_uris else sorted(cluster_nodes, key=str)[0]
        for node in cluster_nodes:
            replace_map[node] = canonical

    # Merge graph

    merged_graph = rdflib.Graph()
    for prefix, uri in g.namespaces():
        merged_graph.bind(prefix, uri)

    for s, p, o in g:
        if p == OWL.sameAs:
            continue
        # Rewire subjects and objects to the canonical ID if a mapping exists
        new_s = replace_map.get(s, s)
        new_o = replace_map.get(o, o)

        merged_graph.add((new_s, p, new_o))

    # 4. Save the new graph to a separate file, keeping the original intact
    merged_graph.serialize(destination=output_path, format="xml")

    return merged_graph



In [ ]:
if __name__ == "__main__":
    DATASET_DIR = Path("EU_DigitalLaw")
    ACTS = ['AI_Act', 'DORA', 'GDPR', 'NIS2']

    for act in ACTS:
        path = DATASET_DIR / act / "metadata.xml"
        output = DATASET_DIR / act / "cleaned_metadata.xml"
        merge_identifiers(path, output)